In [1]:
# Colab에서 실행 (셀 맨 위)
!pip install -q "transformers==4.48.0" timm einops pillow pandas tqdm peft
!pip uninstall -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
"""
Jina-CLIP v2 LoRA 파인튜닝 (LookMatch) — 로컬 / Colab / 구글드라이브 겸용
========================================================================
crop된 옷 이미지 + qwen_caption 쌍으로 Jina-CLIP v2를 도메인 파인튜닝.
대조학습(대칭 InfoNCE)으로 "이 옷 이미지 ↔ 이 캡션"을 정렬.

특징:
  - 환경 자동 감지 (로컬 / Colab / 드라이브)
  - 이미지를 로컬로 복사 후 학습 (드라이브 I/O 병목 제거)
  - LoRA 파인튜닝 (가볍고 빠름)
  - 대칭 InfoNCE 손실 (이미지↔텍스트 양방향)
  - Recall@K 평가 + 최고 성능 저장
  - 체크포인트 (끊겨도 이어서)

입력:  <데이터폴더>/metadata_captioned_final.csv (crop_path, qwen_caption)
출력:  <데이터폴더>/clip_finetuned/  (LoRA 가중치)

설치:
  [로컬]  pip install torch transformers peft pillow pandas tqdm einops timm
  [Colab] !pip install -q transformers peft einops timm pillow pandas tqdm

실행:  python finetune_clip.py

※ 주의:
  - Jina-CLIP v2는 trust_remote_code=True 필요, cc-by-nc-4.0 라이선스(비상업).
  - encode_text/encode_image는 gradient가 흐르므로 파인튜닝 가능.
  - 처음엔 SMOKE_TEST=True로 소량 검증 후 전체 실행 권장.
"""

import os
import gc
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm


# ============================================================
# 경로 설정 — 로컬 / Colab / 드라이브 자동 감지
# ============================================================
def setup_data_dir():
    env_dir = os.environ.get("LOOKMATCH_DIR")
    if env_dir:
        print(f"데이터 경로(환경변수): {env_dir}")
        return Path(env_dir)
    try:
        import google.colab  # noqa
        is_colab = True
    except ImportError:
        is_colab = False
    if is_colab:
        drive_root = Path("/content/drive/MyDrive")
        if not drive_root.exists():
            print("구글 드라이브 연결 중...")
            from google.colab import drive
            drive.mount("/content/drive")
        data_dir = drive_root / "clothes_dataset"
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"데이터 경로(구글드라이브): {data_dir}")
        return data_dir, True
    data_dir = Path("clothes_dataset")
    print(f"데이터 경로(로컬): {data_dir.resolve()}")
    return data_dir, False


_result = setup_data_dir()
if isinstance(_result, tuple):
    DATA_DIR, IS_COLAB = _result
else:
    DATA_DIR, IS_COLAB = _result, False

# 입력 CSV: final 우선, 없으면 captioned
IN_CSV = DATA_DIR / "metadata_captioned_final.csv"
if not IN_CSV.exists():
    for alt in ["metadata_captioned.csv", "metadata_captioned_test_final.csv",
                "metadata_captioned_test.csv"]:
        if (DATA_DIR / alt).exists():
            IN_CSV = DATA_DIR / alt
            break

OUT_DIR = str(DATA_DIR / "clip_finetuned")
CROPS_SUBDIR = "crops"
# Colab이면 이미지를 로컬로 복사해 I/O 병목 제거
LOCAL_CROPS = Path("/content/crops_local") if IS_COLAB else (DATA_DIR / CROPS_SUBDIR)

# ============================================================
# 하이퍼파라미터
# ============================================================
MODEL_NAME = "jinaai/jina-clip-v2"
TRUNCATE_DIM = 512       # Matryoshka 차원 (512면 저장·속도 이득, None이면 1024)
BATCH_SIZE = 4           # 512 해상도라 크게 못 잡음 (OOM 시 4로 더 줄이기)
EPOCHS = 3
LR = 1e-5
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
VAL_RATIO = 0.1
SEED = 42
SMOKE_TEST = False        # True면 소량(200개)으로 빠르게 검증. 검증되면 False로.
SMOKE_N = 200
# True면 이미지를 로컬로 복사하지 않고 드라이브에서 직접 읽음.
# 데이터가 많아 최초 복사가 오래 걸릴 때 유용 (대신 매 epoch I/O는 다소 느림).
SKIP_LOCAL_COPY = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def resolve_image_path(crop_path):
    """윈도우/리눅스 경로 혼용 처리 + 로컬 복사본 우선"""
    fname = str(crop_path).replace("\\", "/").split("/")[-1]
    # 로컬 복사본 우선
    local = LOCAL_CROPS / fname
    if local.exists():
        return str(local)
    # 원본 경로
    if Path(crop_path).exists():
        return str(crop_path)
    cand = DATA_DIR / CROPS_SUBDIR / fname
    if cand.exists():
        return str(cand)
    return None


# ============================================================
# 이미지 로컬 복사 (Colab I/O 병목 제거)
# ============================================================
def copy_images_local(df):
    """드라이브 crops → 로컬로 복사 (Colab에서 학습 속도 대폭 향상)"""
    if not IS_COLAB or SKIP_LOCAL_COPY:
        if SKIP_LOCAL_COPY:
            print("SKIP_LOCAL_COPY=True → 로컬 복사 건너뛰고 드라이브에서 직접 읽습니다.")
        return
    LOCAL_CROPS.mkdir(parents=True, exist_ok=True)
    src_dir = DATA_DIR / CROPS_SUBDIR
    print(f"이미지를 로컬로 복사 중... ({src_dir} → {LOCAL_CROPS})")
    copied = 0
    for crop_path in tqdm(df["crop_path"], desc="복사"):
        fname = str(crop_path).replace("\\", "/").split("/")[-1]
        dst = LOCAL_CROPS / fname
        if dst.exists():
            continue
        src = src_dir / fname
        if src.exists():
            try:
                shutil.copy(src, dst)
                copied += 1
            except Exception:
                pass
    print(f"복사 완료: {copied}개")


# ============================================================
# 데이터셋
# ============================================================
class ClothesDataset(Dataset):
    def __init__(self, df):
        self.rows = []
        for _, row in df.iterrows():
            img = resolve_image_path(row["crop_path"])
            cap = str(row.get("qwen_caption", "")).strip()
            if img and cap:
                self.rows.append((img, cap))
        print(f"  유효 샘플: {len(self.rows)}개")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        img_path, caption = self.rows[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (512, 512), (255, 255, 255))
        return {"image": image, "caption": caption}


def collate_fn(batch):
    return {
        "images": [b["image"] for b in batch],
        "captions": [b["caption"] for b in batch],
    }


# ============================================================
# 모델 로드 + LoRA
# ============================================================
def load_model():
    from transformers import AutoModel, AutoConfig
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"디바이스: {device}")
    print(f"모델 로드: {MODEL_NAME} (처음이면 다운로드에 시간 걸림)")

    # config에서 flash attention / xformers 비활성화 (설치 안 된 환경 대응)
    config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
    # 핵심: attention 구현을 eager(순수 PyTorch)로 → rotary backward의 FlashAttn 강제 호출 우회
    config._attn_implementation = "eager"
    for attr in ["use_text_flash_attn", "use_vision_xformers"]:
        if hasattr(config, attr):
            setattr(config, attr, False)
    for sub in ["text_config", "vision_config"]:
        if hasattr(config, sub):
            subcfg = getattr(config, sub)
            # 하위 config에도 eager 지정
            setattr(subcfg, "_attn_implementation", "eager")
            for attr in ["use_flash_attn", "use_text_flash_attn",
                         "use_vision_xformers", "x_attention"]:
                if hasattr(subcfg, attr):
                    setattr(subcfg, attr, False)

    model = AutoModel.from_pretrained(
        MODEL_NAME, trust_remote_code=True, config=config
    ).to(device)

    # ── 텍스트 인코더 완전 freeze ──
    # 이유: 텍스트(xlm-roberta-flash)의 rotary backward가 FlashAttention을
    #       강제 호출 → 미설치 환경에선 학습 시 에러. 텍스트를 학습에서 제외하면
    #       gradient가 안 흘러 rotary backward가 호출되지 않음.
    #       Qwen 캡션은 여전히 "정답 기준"으로 쓰이므로 가치는 유지됨.
    for name, p in model.named_parameters():
        p.requires_grad = False   # 일단 전체 freeze

    # ── 비전 인코더에만 LoRA 적용 ──
    try:
        from peft import LoraConfig, get_peft_model
        # vision_model. 로 시작하는 nn.Linear 중 attention/mlp 계층만 정확히 타깃
        # (텍스트 모듈은 제외 → rotary FlashAttention 회피)
        vision_targets = []
        for name, module in model.named_modules():
            if name.startswith("vision_model.") and isinstance(module, torch.nn.Linear):
                if any(name.endswith(s) for s in
                       ["q_proj", "k_proj", "v_proj", "proj", "w1", "w2", "w3", "fc1", "fc2"]):
                    vision_targets.append(name)
        print(f"비전 LoRA 대상 모듈: {len(vision_targets)}개")
        if not vision_targets:
            print("[경고] 비전 LoRA 대상을 못 찾음. 모듈 이름 확인 필요.")

        lora_cfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=vision_targets,   # 정확한 전체 경로 (비전만)
            bias="none",
        )
        model = get_peft_model(model, lora_cfg)
        model.print_trainable_parameters()
    except Exception as e:
        print(f"[LoRA 경고] {e}")
        print("→ 비전 LoRA 실패. 텍스트는 freeze 상태 유지.")

    return model, device


def get_base(model):
    """peft 래핑 여부와 무관하게 encode 메서드 접근"""
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        return model.base_model.model
    return model


# ============================================================
# 임베딩 (gradient 흐름 유지) — get_*_features 사용
# ============================================================
_PROC_CACHE = {}
def _get_processors():
    """토크나이저·이미지 프로세서 로드 (한 번만)"""
    if "tokenizer" not in _PROC_CACHE:
        from transformers import AutoTokenizer, AutoImageProcessor
        _PROC_CACHE["tokenizer"] = AutoTokenizer.from_pretrained(
            MODEL_NAME, trust_remote_code=True)
        _PROC_CACHE["image_processor"] = AutoImageProcessor.from_pretrained(
            MODEL_NAME, trust_remote_code=True, use_fast=True)
    return _PROC_CACHE


def encode_batch(model, images, captions, device):
    """gradient가 흐르는 임베딩.
    get_text_features / get_image_features 사용 (forward 계열, gradient 유지)."""
    base = get_base(model)
    proc = _get_processors()

    # 텍스트 토크나이즈
    text_inputs = proc["tokenizer"](
        captions, return_tensors="pt", padding=True, truncation=True, max_length=77
    ).to(device)
    # 이미지 전처리
    image_inputs = proc["image_processor"](
        images=images, return_tensors="pt"
    ).to(device)
    pixel_values = image_inputs["pixel_values"]

    # gradient 유지되는 특징 추출
    txt_emb = base.get_text_features(input_ids=text_inputs["input_ids"])
    img_emb = base.get_image_features(pixel_values=pixel_values)

    # Matryoshka: 앞 TRUNCATE_DIM 차원만
    if TRUNCATE_DIM and txt_emb.size(-1) > TRUNCATE_DIM:
        txt_emb = txt_emb[..., :TRUNCATE_DIM]
        img_emb = img_emb[..., :TRUNCATE_DIM]

    return img_emb.float(), txt_emb.float()


def clip_loss(img_emb, txt_emb, temperature=0.07):
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)
    logits = (img_emb @ txt_emb.t()) / temperature
    labels = torch.arange(len(img_emb), device=img_emb.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2


@torch.no_grad()
def evaluate(model, loader, device, k=5):
    model.eval()
    all_i, all_t = [], []
    for b in loader:
        ie, te = encode_batch(model, b["images"], b["captions"], device)
        # GPU 메모리 누적 방지: 정규화 후 CPU로 이동
        all_i.append(F.normalize(ie, dim=-1).cpu())
        all_t.append(F.normalize(te, dim=-1).cpu())
        del ie, te
        if device == "cuda":
            torch.cuda.empty_cache()
    if not all_i:
        return 0.0
    ie, te = torch.cat(all_i), torch.cat(all_t)
    sims = te @ ie.t()
    topk = sims.topk(min(k, sims.size(1)), dim=1).indices
    correct = sum(i in topk[i] for i in range(len(topk)))
    return correct / len(topk)


# ============================================================
# 학습
# ============================================================
def train():
    print(f"입력 CSV: {IN_CSV}")
    if not IN_CSV.exists():
        print("[오류] 입력 CSV 없음. generate_captions.py → clean_captions.py 먼저 실행.")
        return

    df = pd.read_csv(IN_CSV)
    df = df[df["qwen_caption"].notna() & (df["qwen_caption"].astype(str).str.strip() != "")]
    if SMOKE_TEST:
        df = df.head(SMOKE_N).copy()
        print(f"[SMOKE TEST] {len(df)}개로 빠른 검증 (검증되면 SMOKE_TEST=False)")
    print(f"학습 대상: {len(df)}개")

    # 이미지 로컬 복사 (Colab I/O 병목 제거)
    copy_images_local(df)

    # 분할
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_val = max(1, int(len(df) * VAL_RATIO))
    val_df, train_df = df[:n_val], df[n_val:]
    print("학습셋:", end=" "); train_ds = ClothesDataset(train_df)
    print("검증셋:", end=" "); val_ds = ClothesDataset(val_df)

    # 드라이브 직접 읽기(SKIP_LOCAL_COPY)면 I/O 대기가 병목이라 worker를 늘려
    # 여러 이미지를 동시에 읽게 함
    n_workers = 8 if (IS_COLAB and SKIP_LOCAL_COPY) else 2
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=n_workers)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=n_workers)

    model, device = load_model()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR
    )

    print(f"\n학습 시작 (epochs={EPOCHS}, batch={BATCH_SIZE})\n")
    best = 0.0
    for epoch in range(EPOCHS):
        model.train()
        total = 0.0
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for b in pbar:
            try:
                optimizer.zero_grad()
                ie, te = encode_batch(model, b["images"], b["captions"], device)
                if not ie.requires_grad:
                    print("\n[경고] 임베딩에 gradient가 없습니다. "
                          "get_*_features 경로를 확인하세요.")
                loss = clip_loss(ie, te)
                loss.backward()
                optimizer.step()
                total += loss.item()
                pbar.set_postfix(loss=f"{loss.item():.4f}")
                del ie, te, loss
            except torch.cuda.OutOfMemoryError:
                print(f"\n[OOM] 배치 건너뜀. BATCH_SIZE를 더 줄이세요 (현재 {BATCH_SIZE}).")
                optimizer.zero_grad()
                if device == "cuda":
                    torch.cuda.empty_cache()
                continue

        recall = evaluate(model, val_loader, device, k=5)
        dt = time.time() - t0
        print(f"  Epoch {epoch+1}: loss={total/max(len(train_loader),1):.4f}, "
              f"Recall@5={recall:.3f}, {dt:.0f}초")

        if recall >= best:
            best = recall
            Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
            try:
                model.save_pretrained(OUT_DIR)
                print(f"  → 저장: {OUT_DIR} (Recall@5={recall:.3f})")
            except Exception as e:
                print(f"  저장 경고: {e}")

        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    print(f"\n{'='*55}")
    print(f"학습 완료! 최고 Recall@5 = {best:.3f}")
    print(f"LoRA 가중치: {OUT_DIR}")
    if SMOKE_TEST:
        print("\n※ SMOKE TEST 완료. 정상이면 SMOKE_TEST=False로 바꿔 전체 학습하세요.")
    print(f"{'='*55}")


if __name__ == "__main__":
    train()

데이터 경로(구글드라이브): /content/drive/MyDrive/clothes_dataset
입력 CSV: /content/drive/MyDrive/clothes_dataset/metadata_captioned_final.csv
학습 대상: 71637개
SKIP_LOCAL_COPY=True → 로컬 복사 건너뛰고 드라이브에서 직접 읽습니다.
학습셋:   유효 샘플: 64474개
검증셋:   유효 샘플: 7163개
디바이스: cuda
모델 로드: jinaai/jina-clip-v2 (처음이면 다운로드에 시간 걸림)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- configuration_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

  Epoch 1: loss=0.0590, Recall@5=0.460, 6495초
  → 저장: /content/drive/MyDrive/clothes_dataset/clip_finetuned (Recall@5=0.460)


Epoch 2/3: 100%|██████████| 16119/16119 [1:40:40<00:00,  2.67it/s, loss=0.0442]


  Epoch 2: loss=0.0413, Recall@5=0.480, 6446초
  → 저장: /content/drive/MyDrive/clothes_dataset/clip_finetuned (Recall@5=0.480)


Epoch 3/3: 100%|██████████| 16119/16119 [1:40:48<00:00,  2.66it/s, loss=0.0000]


  Epoch 3: loss=0.0373, Recall@5=0.493, 6454초
  → 저장: /content/drive/MyDrive/clothes_dataset/clip_finetuned (Recall@5=0.493)

학습 완료! 최고 Recall@5 = 0.493
LoRA 가중치: /content/drive/MyDrive/clothes_dataset/clip_finetuned
